In [ ]:
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import davies_bouldin_score as davies
# allow importing the "backbone" package when the kernel's cwd is noteonooks/
sys.path.append(os.path.abspath(".."))
import backbone.data_handle.Distributions as dist
import backbone.data_handle.Test as test
import backbone.custom_metrics.AstroMLmodified as AstroMLmod
import backbone.custom_metrics.AstroMLmod3 as am3
import backbone.visuals.VISUAL as viz

random.seed(42)

# Gaussian Zoo generator

## Below is code talored for generating feature spaces of arbitrary dimensions, made of data points generated from gaussian distributions. It allows for varying the the number of gaussians, the covariances of the generated clusters, individual means of the gaussians. It also allows for comparing clustering accuracy, structure content as defined by the davies bouldin, and finally the 2-point correlation (2PCF) function as well as the 2PCF score. 

## The 2PCF as when varying cluster cohession:

In [ ]:
#saving metrics for comparing
norm_scores_cov = []
cos_scores_cov = []
davies_scores_cov = []
sim_scores_cov = []
acc_cov = []

#parameters for feature spaces

scatter = False
dimensions = 2 
num_samples = 2
points_per_gaussian = 2500
labels = []
cov_ranges  = [80,160,240,320,400,480,640,730,800,1000,1500,2000]

#Creating labels for classification
plt.style.use("science")
plt.figure(figsize=(15,10))
plt.rcParams.update({'font.size': 40})



for i in range(num_samples):
    for item in range(points_per_gaussian):
        labels.append(i)
        
        
for cov_diagonal in cov_ranges:

    means = [[-50,0],[50,0]]
    covariances = dist.symmetric_covariance(cov_diagonal,
                                            dimension = dimensions)

    #Ceatures the feature space one cluster at a time from means and covariances
        
    gaussian_space = dist.generate_gaussian_points(mean = means[0],
                                                   cov = covariances[0],
                                                   n_points =points_per_gaussian,
                                                   seed = 42)  # places cluster at position "mean"
    for mean in means[1:]:
            
        gaussian_space = np.concatenate((gaussian_space,
                                         dist.generate_gaussian_points(mean, covariances[0],points_per_gaussian,seed = 42)),
                                        axis = 0)


    #clustering

    k_labels = dist.kmeans(gaussian_space,
                           n_clusters = 2)

    #scores
    
    davies_ = davies(gaussian_space,k_labels)
    val = test.KNN_accuracy(gaussian_space,labels)
    sim_score = viz.similarity_index(labels,k_labels)

    corr, dcorr, NormScore = AstroMLmod.correlate_and_plot(gaussian_space,
                                                        min_dist = 0.0,
                                                        max_dist =1.5,
                                                        label = "Correlation on flat manifold for epoch:"+str(cov_diagonal),
                                                        fig_name = "plots/2PCR@Epoch: "+str(cov_diagonal),
                                                        precomputed_RR = None,
                                                        bin_number = 100,
                                                        method = "standard",
                                                        Nbootstrap  = 5,
                                                        plot = False,
                                                        return_corr = True,
                                                        background_factor = 1,
                                                        representations = [])
    cos_score = am3.TPCF_score(gaussian_space, method = "standard")

    min_dist, max_dist, bin_number = 0,0.68,100
    bins = np.linspace(min_dist,
                       max_dist, 
                       bin_number)

    #metrics
    norm_scores_cov.append(NormScore)
    cos_scores_cov.append(cos_score)
    acc_cov.append(val)
    davies_scores_cov.append(davies_)
    sim_scores_cov.append(sim_score)


    if scatter: 
        
        ax = viz.shade(name = "$\sigma_{xx} = \sigma_{yy}  = $"+" "+str(cov_diagonal)+"; "+ "$|u1- u2|  = $"+" "+str(100),
                  predictions = labels,
                  numof_class = len(means),
                  embedded_dataset = gaussian_space,
                  alpha = 0.6,
                  size = 50,
                  save = False,
                  hard_coloring = True,
                )

        ax.set_xlim(-100, 100)
        ax.set_ylim(-100, 100)
        ax.plot(means[0][0], means[0][1], 'x', color='black', markersize=30,markeredgewidth=10)
        ax.plot(means[1][0], means[1][1], 'x', color='black', markersize=30, markeredgewidth=10)
        if cov_diagonal in [80,640]:
            fig = ax.get_figure()
            fig.savefig(str(cov_diagonal)+"sigma_scatter.pdf")
        plt.show()

        
    else:
        if cov_diagonal in [80,160,320,640]:
            line, = plt.plot(bins[1:],corr,
                                 label = "$\sigma_{xx}$: "+ str(cov_diagonal))
            plt.fill_between(bins[1:],corr- dcorr, corr+dcorr, color = line.get_color(),alpha = .3)
    
plt.axhline(0, color='gray', linestyle='--', linewidth=1,label = "y = 0") 
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.ylabel("$\epsilon(r)$")
plt.xlabel("Distance scale (r)")
plt.gcf().savefig("cov_change_overlay.pdf")
plt.show()

# Metrics over covariance changes

In [ ]:
plt.style.use("science")
plt.rcParams.update({'font.size': 25}) 
fig, axs = plt.subplots(4, 1, sharex=True, figsize=(15, 20))


x = [np.sqrt(cov) for cov in cov_ranges]
yerr = np.array([b for a,b in norm_scores_cov])
y_cov = np.array([a for a,b in  norm_scores_cov])
y_cos_cov = np.array([a for a,b in cos_scores_cov])

axs[0].plot(x, y_cov,color = "C0",marker = "o",alpha = 1,label='TPCF score')

axs[0].errorbar(x, y_cov, yerr = yerr, color = "C0",alpha = 1)

axs[0].set_ylabel('2PCF score')
axs[0].legend()


axs[1].plot(x, y_cos_cov,color = "C3",marker = "o",alpha = 1,label='Cosine TPCF score')
axs[1].errorbar(x, y_cos_cov, yerr = [b for a,b in cos_scores_cov], color = "C3",alpha = 1)
axs[1].set_ylabel('Cosine 2PCF score')
axs[1].legend()


axs[2].plot(x, [a for a in davies_scores_cov],color = "C1",marker = "o",alpha = 1,label='Davies-bouldin score')


axs[2].set_ylabel('Davies-bouldin score')
axs[2].legend()


axs[3].plot(x,[a for a,b in acc_cov], marker = "o",alpha = 1,label='KNN accuracy', color = "C2")
axs[3].errorbar(x, [a for a,b in acc_cov],yerr = [b for a,b in  acc_cov], color = "C2",alpha = 1)
axs[3].set_ylabel('KNN accuracy')
axs[3].set_xlabel('Intra-cluster separation ($\sigma_{xx}$)')
axs[3].legend()

# Adjust spacing
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Show plot
plt.savefig("IC_separation.pdf")
plt.show()

# Gaussian Zoo generator

## Cluster separation

In [ ]:
plt.style.use("science")
plt.figure(figsize=(15,10))
plt.rcParams.update({'font.size': 40})


norm_scores_sep = []
cos_scores_sep = []
davies_scores_sep =[]
acc = []
sim_scores_sep = []

scatter = True
dimensions = 2  
num_samples = 2 
points_per_gaussian = 2500

cov_diagonal = 80
covariances = dist.symmetric_covariance(cov_diagonal,
                                        dimension = dimensions)
mean_separation =  [10,8,6,4,3,2,1]
labels = []


for i in range(num_samples):
    for item in range(points_per_gaussian):
        labels.append(i)

    


for separation in mean_separation:
    separation = separation**2
    
    means = np.array([[separation,0],[-separation,0]])
    
    #Ceatures the feature space one cluster at a time from means and covariances

    gaussian_space = dist.generate_gaussian_points(mean = means[0],
                                                   cov = covariances[0],
                                                   n_points =points_per_gaussian,
                                                   seed = 52)  # places cluster at position "mean"

    for mean in means[1:]:
            
        gaussian_space = np.concatenate((gaussian_space,
                                         dist.generate_gaussian_points(means[1],
                                                                       covariances[0],
                                                                       points_per_gaussian,seed = 42)),
                                        axis = 0)


    #clustering

    k_labels = dist.kmeans(gaussian_space,
                           n_clusters = 2)

    #scores
    
    davies_ = davies(gaussian_space,k_labels)
    val = test.KNN_accuracy(gaussian_space,labels)
    sim_score = viz.similarity_index(labels,k_labels)

    corr, dcorr, NormScore = AstroMLmod.correlate_and_plot(gaussian_space,
                                                        min_dist = 0.0,
                                                        max_dist =1.5,
                                                        label = "Correlation on flat manifold for epoch:"+str(cov_diagonal),
                                                        fig_name = "plots/2PCR@Epoch: "+str(cov_diagonal),
                                                        precomputed_RR = None,
                                                        bin_number = 100,
                                                        method = "standard",
                                                        bootstrap  = True,
                                                        plot = False,
                                                        return_corr = True,
                                                        background_factor = 1,
                                                        representations = [])
    cos_score = am3.TPCF_score(gaussian_space, method = "standard")

    min_dist, max_dist, bin_number = 0,0.68,100
    bins = np.linspace(min_dist,
                       max_dist, 
                       bin_number)
    #metrics
    norm_scores_sep.append(NormScore)
    cos_scores_sep.append(cos_score)
    acc.append(val)
    davies_scores_sep.append(davies_)
    sim_scores_sep.append(sim_score)


    if scatter: 
        
        ax = viz.shade(name = "$\sigma_{xx} = \sigma_{yy}  = $"+" "+str(cov_diagonal)+"; "+ "$|u_1-u_2| = $"+str(separation*2),
                  predictions = labels,
                  numof_class = len(means),
                  embedded_dataset = gaussian_space,
                  alpha = 0.6,
                  size = 50,
                  save = False,
                  hard_coloring = True,
                      )

        ax.set_xlim(-100, 100)
        ax.set_ylim(-100, 100)
        ax.plot(means[0][0], means[0][1], 'x', color='black', markersize=30,markeredgewidth=10)
        ax.plot(means[1][0], means[1][1], 'x', color='black', markersize=30, markeredgewidth=10)
        
        if separation in [32,16]:
            fig = ax.get_figure()
            fig.savefig(str(separation*2)+"separation_scatter.pdf")
        plt.show()

    else:
        if 2*separation in [128,72,32,8]:
            line, = plt.plot(bins[1:],corr,
                                 label = "$|u1-u2|$: "+ str(2*separation))
            plt.fill_between(bins[1:],corr- dcorr, corr+dcorr, color = line.get_color(),alpha = .3)
    
plt.axhline(0, color='gray', linestyle='--', linewidth=1,label = "y = 0") 
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.ylabel("$\epsilon(r)$")
plt.xlabel("Distance scale (r)")
plt.gcf().savefig("sep_change_overlay.pdf")
plt.show()

In [ ]:
plt.style.use("science")
plt.rcParams.update({'font.size': 25}) 
fig, axs = plt.subplots(4, 1, sharex=True, figsize=(15, 20))


x = [sep for sep in  mean_separation]
yerr = np.array([b for a,b in norm_scores_sep])
y_sep = np.array([a for a,b in  norm_scores_sep])
y_cos_sep = np.array([a for a,b in cos_scores_sep])

axs[0].plot(x, y_sep,color = "C0",marker = "o",alpha = 1,label='2PCF score')

axs[0].errorbar(x, y_sep, yerr = yerr, color = "C0",alpha = 1)

axs[0].set_ylabel('2PCF score')
axs[0].legend()


axs[1].plot(x, y_cos_sep,color = "C3",marker = "o",alpha = 1,label='Cosine 2PCF score')
axs[1].errorbar(x, y_cos_sep, yerr = [b for a,b in cos_scores_sep], color = "C3",alpha = 1)
axs[1].set_ylabel('Cosine 2PCF score')
axs[1].legend()


axs[2].plot(x, [a for a in davies_scores_sep],color = "C1",marker = "o",alpha = 1,label='Davies-bouldin score')


axs[2].set_ylabel('Davies-Bouldin score')
axs[2].legend()


axs[3].plot(x,[a for a,b in acc], marker = "o",alpha = 1,label='KNN accuracy', color = "C2")
axs[3].errorbar(x, [a for a,b in acc],yerr = [b for a,b in  acc], color = "C2",alpha = 1)
axs[3].set_ylabel('KNN accuracy')
axs[3].set_xlabel('Cluster mean separation $|U_1 - U_2|$')
axs[3].legend()

# Adjust spacing
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Show plot
plt.savefig("ICM_separation.pdf")
plt.show()

## The 2PCF score correlates with other clustering metrics

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(25, 20),dpi = 300)

plt.style.use("science")
plt.rcParams.update({'font.size': 25}) 
ax1.scatter(y_cov,[a for a in davies_scores_cov], label = "Varying cluster cohesion")
ax1.scatter(y_sep,[a for a in davies_scores_sep], label = "Varying cluster separation")
ax1.set_xlabel("TPCF score")
ax1.set_ylabel("Davies-Boulding score")
ax1.legend()


ax2.scatter(y_cov,[a for a in sim_scores_cov])
ax2.scatter(y_sep,[a for a in sim_scores_sep])


ax2.set_xlabel("TPCF score")
ax2.set_ylabel("Clustering accuracy")

ax3.scatter(y_cos_cov,[a for a in davies_scores_cov])
ax3.scatter(y_cos_sep,[a for a in davies_scores_sep])
ax3.set_xlabel("Cosine TPCF score")
ax3.set_ylabel("Davies-Boulding score")

ax4.scatter(y_cos_cov,[a for a in sim_scores_cov])
ax4.scatter(y_cos_sep,[a for a in sim_scores_sep])
ax4.set_xlabel("Cosine TPCF score")
ax4.set_ylabel("Clustering accuracy")

plt.tight_layout()
plt.savefig("correlation_between_metrics.pdf")

In [ ]:
fig = plt.figure(dpi = 300, figsize = (15,15))
plt.style.use("science")
plt.rcParams.update({'font.size': 25}) 
plt.scatter([a for a in davies_scores_cov], [a for a in sim_scores_cov],label = "Varying cluster cohesion")
plt.scatter([a for a in davies_scores_sep],[a for a in sim_scores_sep], label = "Varying cluster separation")
plt.savefig("correlation_between_metrics.pdf")

In [ ]:
#saving metrics for comparing
norm_scores_cov= []
davies_scores_cov = []
sim_scores_cov = []
acc_cov = []

#parameters for feature spaces

scatter = False
dimensions = 2 
num_samples = 2

# FIXED: Keep covariance and mean separation constant
fixed_cov_diagonal = 320  # Fixed covariance value
fixed_means = [[-50, 0], [50, 0]]  # Fixed mean separation of 100

# VARYING: Number of points per cluster
points_per_cluster_values = [100, 250, 500, 1000, 1500, 2000, 2500, 3000, 4000, 5000]

plt.style.use("science")
plt.figure(figsize=(15,10))
plt.rcParams.update({'font.size': 40})

for points_per_gaussian in points_per_cluster_values:
    
    # Recalculate labels for each iteration
    labels = []
    for i in range(num_samples):
        for item in range(points_per_gaussian):
            labels.append(i)
    
    # Use fixed means and covariances
    means = fixed_means
    covariances = dist.symmetric_covariance(fixed_cov_diagonal,
                                            dimension=dimensions)

    # Creates the feature space one cluster at a time from means and covariances
    gaussian_space = dist.generate_gaussian_points(mean=means[0],
                                                   cov=covariances[0],
                                                   n_points=points_per_gaussian,
                                                   seed=42)  # places cluster at position "mean"
    for mean in means[1:]:
        gaussian_space = np.concatenate((gaussian_space,
                                         dist.generate_gaussian_points(mean, covariances[0], points_per_gaussian, seed=42)),
                                        axis=0)

    # clustering
    k_labels = dist.kmeans(gaussian_space,
                           n_clusters=2)

    # scores
    davies_ = davies(gaussian_space, k_labels)
    val = test.KNN_accuracy(gaussian_space, labels)
    sim_score = viz.similarity_index(labels, k_labels)
    
    corr, dcorr, NormScore = AstroMLmod.correlate_and_plot(gaussian_space,
                                                        min_dist=0.0,
                                                        max_dist=1.5,
                                                        label="Correlation on flat manifold for n_points:" + str(points_per_gaussian),
                                                        fig_name="plots/2PCR@n_points: " + str(points_per_gaussian),
                                                        precomputed_RR=None,
                                                        bin_number=100,
                                                        method="standard",
                                                        Nbootstrap=5,
                                                        plot=False,
                                                        return_corr=True,
                                                        background_factor=1,
                                                        representations=[])

    min_dist, max_dist, bin_number = 0, 0.68, 100
    bins = np.linspace(min_dist,
                       max_dist, 
                       bin_number)

    # metrics
    norm_scores_cov.append(NormScore)
    acc_cov.append(val)
    davies_scores_cov.append(davies_)
    sim_scores_cov.append(sim_score)

    if scatter: 
        ax = viz.shade(name="$n_{points} = $" + " " + str(points_per_gaussian) + "; " + "$|u1- u2| = $" + " " + str(100),
                  predictions=labels,
                  numof_class=len(means),
                  embedded_dataset=gaussian_space,
                  alpha=0.6,
                  size=50,
                  save=False,
                  hard_coloring=True,
                )

        ax.set_xlim(-100, 100)
        ax.set_ylim(-100, 100)
        ax.plot(means[0][0], means[0][1], 'x', color='black', markersize=30, markeredgewidth=10)
        ax.plot(means[1][0], means[1][1], 'x', color='black', markersize=30, markeredgewidth=10)
        if points_per_gaussian in [100, 2500]:
            fig = ax.get_figure()
            fig.savefig(str(points_per_gaussian) + "points_scatter.pdf")
        plt.show()
    else:
        if points_per_gaussian in [100, 500, 1000, 2500, 5000]:
            line, = plt.plot(bins[1:], corr,
                           label="$n_{points}$: " + str(points_per_gaussian))
            plt.fill_between(bins[1:], corr - dcorr, corr + dcorr, color=line.get_color(), alpha=.3)
    
plt.axhline(0, color='gray', linestyle='--', linewidth=1, label="y = 0") 
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.ylabel("$\epsilon(r)$")
plt.xlabel("Distance scale (r)")
plt.gcf().savefig("points_change_overlay.pdf")
plt.show()

In [ ]:
plt.plot([a for a,b in norm_scores_cov])
plt.show()